# PReLU & Learnable Activations



## 1. Learning Objectives

- Understand why ReLU fails.
- Learn PReLU.
- Implement PReLU from scratch.
- Compare ReLU, LeakyReLU, PReLU and GELU.

## 2. Evolution

Sigmoid → Tanh → ReLU → LeakyReLU → **PReLU** → ELU → SELU → GELU → Swish → Mish → Dynamic ReLU

## 3. First Principles

Instead of humans fixing the negative slope, let the network **learn** it.

## 4. Mathematical Formulation


ReLU:

\[
f(x)=\max(0,x)
\]

LeakyReLU:

\[
f(x)=
\begin{cases}
x,&x>0\\
\alpha x,&x<0
\end{cases}
\]

PReLU:

\[
f(x)=
\begin{cases}
x,&x>0\\
\alpha x,&x<0
\end{cases}
\]

where \(\alpha\) is trainable.


## 5. Gradients


Positive:

\[
\frac{\partial y}{\partial x}=1
\]

Negative:

\[
\frac{\partial y}{\partial x}=\alpha
\]

Parameter gradient:

\[
\frac{\partial y}{\partial \alpha}=x
\]


## 6. Advantages

- Learnable activation
- Reduces dead neurons
- Tiny parameter overhead

## 7. Limitations

- Slightly more parameters
- Usually not used in Transformers

## 8. Research Papers

- He et al. (2015) Delving Deep into Rectifiers
- Glorot et al. (2011)
- GELU (2016)
- Swish (2017)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x=np.linspace(-5,5,500)
relu=np.maximum(0,x)
leaky=np.where(x>0,x,0.01*x)
prelu=np.where(x>0,x,0.25*x)

plt.figure(figsize=(8,5))
plt.plot(x,relu,label='ReLU')
plt.plot(x,leaky,label='LeakyReLU')
plt.plot(x,prelu,label='PReLU')
plt.grid(); plt.legend(); plt.show()


In [ ]:
class PReLU_Numpy:
    def __init__(self,alpha=0.25):
        self.alpha=alpha
    def forward(self,x):
        return np.where(x>0,x,self.alpha*x)

layer=PReLU_Numpy()
print(layer.forward(np.array([-3,-1,0,2])))


In [ ]:
import torch
import torch.nn as nn

layer=nn.PReLU(num_parameters=1,init=0.25)
x=torch.tensor([-2.,-1.,0.,1.,2.])
print(layer(x))


In [ ]:
import torch
import torch.nn as nn

class MyPReLU(nn.Module):
    def __init__(self,alpha=0.25):
        super().__init__()
        self.alpha=nn.Parameter(torch.tensor(alpha))
    def forward(self,x):
        return torch.where(x>0,x,self.alpha*x)

m=MyPReLU()
print(m(torch.tensor([-2.,-1.,0.,1.,2.])))


In [ ]:
class PReLUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(2,16),
            nn.PReLU(),
            nn.Linear(16,16),
            nn.PReLU(),
            nn.Linear(16,1),
            nn.Sigmoid()
        )
    def forward(self,x):
        return self.net(x)

print(PReLUNet())


In [ ]:
# XOR Training
import torch.optim as optim

x=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=torch.tensor([[0.],[1.],[1.],[0.]])

model=PReLUNet()
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.05)

for epoch in range(2000):
    optimizer.zero_grad()
    pred=model(x)
    loss=criterion(pred,y)
    loss.backward()
    optimizer.step()

print(model(x).detach())


## MNIST Mini Project
Compare ReLU, LeakyReLU, PReLU and GELU on the same MLP.

Metrics:
- Accuracy
- Loss
- Training Time
- Parameter Count
- Confusion Matrix

In [ ]:
# Print learned alpha parameters
for name,param in model.named_parameters():
    if "weight" not in name:
        print(name,param.data)


## Comparison

| Activation | Learnable | Dead Neurons | Parameters |
|---|---:|---:|---:|
| ReLU | ❌ | Yes | 0 |
| LeakyReLU | ❌ | No | 0 |
| PReLU | ✅ | No | Few |
| GELU | ❌ | No | 0 |


## Exercises

1. Implement LeakyReLU manually.
2. Implement PReLU manually.
3. Plot ReLU vs LeakyReLU vs PReLU.
4. Train XOR.
5. Train MNIST.
6. Plot learned alpha over epochs.
7. Compare with GELU.
8. Implement backward pass manually.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LeakyReLU:

    def __init__(self, alpha=0.01):
        self.alpha = alpha

    def forward(self, x):
        return np.where(x > 0, x, self.alpha * x)


activation = LeakyReLU(alpha=0.1)

x = np.linspace(-5, 5, 500)
y = activation.forward(x)

plt.plot(x, y)
plt.title("Leaky ReLU")
plt.grid(True)
plt.show()

In [ ]:
import torch
import torch.nn as nn

class MyLeakyReLU(nn.Module):

    def __init__(self, alpha=0.1):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return torch.where(x > 0, x, self.alpha * x)


layer = MyLeakyReLU()

x = torch.tensor([-2., -1., 0., 1., 2.])

print(layer(x))

In [ ]:
import torch
import torch.nn as nn

class MyPReLU(nn.Module):

    def __init__(self, alpha=0.25):
        super().__init__()

        self.alpha = nn.Parameter(
            torch.tensor(alpha)
        )

    def forward(self, x):

        return torch.where(
            x > 0,
            x,
            self.alpha * x
        )


layer = MyPReLU()

x = torch.tensor([-2., -1., 0., 1., 2.])

print(layer(x))

In [ ]:
for name, param in layer.named_parameters():
    print(name, param)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 500)

relu = np.maximum(0, x)

leaky = np.where(x > 0, x, 0.1 * x)

prelu = np.where(x > 0, x, 0.25 * x)

plt.figure(figsize=(8,5))

plt.plot(x, relu, label="ReLU")
plt.plot(x, leaky, label="LeakyReLU")
plt.plot(x, prelu, label="PReLU")

plt.title("Activation Function Comparison")
plt.xlabel("Input")
plt.ylabel("Output")

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

X = torch.tensor([
    [0.,0.],
    [0.,1.],
    [1.,0.],
    [1.,1.]
])

Y = torch.tensor([
    [0.],
    [1.],
    [1.],
    [0.]
])

model = nn.Sequential(

    nn.Linear(2,16),

    nn.PReLU(),

    nn.Linear(16,16),

    nn.PReLU(),

    nn.Linear(16,1),

    nn.Sigmoid()

)

criterion = nn.BCELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.05
)

for epoch in range(2000):

    optimizer.zero_grad()

    pred = model(X)

    loss = criterion(pred,Y)

    loss.backward()

    optimizer.step()

print(model(X).detach())

In [ ]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

train_loader = DataLoader(

    MNIST(
        "data",
        train=True,
        download=True,
        transform=ToTensor()
    ),

    batch_size=128,

    shuffle=True

)

test_loader = DataLoader(

    MNIST(
        "data",
        train=False,
        download=True,
        transform=ToTensor()
    ),

    batch_size=256

)

In [ ]:
class PReLUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Flatten(),

            nn.Linear(784,256),

            nn.PReLU(),

            nn.Linear(256,128),

            nn.PReLU(),

            nn.Linear(128,10)

        )

    def forward(self,x):

        return self.net(x)

model = PReLUNet()

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

for epoch in range(10):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}"
    )

In [ ]:
alpha_history = []

for epoch in range(10):

    model.train()

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    alpha = model.net[2].weight.item()

    alpha_history.append(alpha)

print(alpha_history)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

plt.plot(alpha_history, marker="o")

plt.title("Learned Alpha")

plt.xlabel("Epoch")

plt.ylabel("Alpha")

plt.grid(True)

plt.show()

| Metric        | PReLU           | GELU            |
| ------------- | --------------- | --------------- |
| Accuracy      | ?               | ?               |
| Training Loss | ?               | ?               |
| Parameters    | Slightly Higher | Lower           |
| Speed         | Fast            | Slightly Slower |
| Dead Neurons  | No              | No              |


In [ ]:
import numpy as np

def prelu_forward(x, alpha):
    return np.where(x > 0, x, alpha * x)

def prelu_backward(x, alpha):
    dx = np.where(x > 0, 1.0, alpha)
    dalpha = np.where(x > 0, 0.0, x)
    return dx, dalpha

x = np.array([-2., -1., 0., 1., 2.])
alpha = 0.25

y = prelu_forward(x, alpha)
dx, dalpha = prelu_backward(x, alpha)

print("Forward:", y)
print("dy/dx:", dx)
print("dy/dalpha:", dalpha)

In [ ]:
import torch

alpha = torch.tensor(0.25, requires_grad=True)
x = torch.tensor([-2., -1., 0., 1., 2.])

y = torch.where(x > 0, x, alpha * x)
loss = y.sum()

loss.backward()

print("Autograd alpha gradient:", alpha.grad)